In [ ]:
"""
=============================================================
INSURANCE RECOMMENDATION PIPELINE
LightFM + LightGBM Ensemble
=============================================================

Arhitektura:
    1. LightFM  → kolaborativni signal + demografija (embedding pristup)
    2. LightGBM → nelinearne kombinacije featuresova (tree pristup)
    3. Ensemble → weighted average oba score-a → finalni ranking

Instalacija:
    pip install lightfm lightgbm pandas numpy scikit-learn
=============================================================
"""

import pandas as pd
import numpy as np
from datetime import datetime
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, auc_score
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


# =============================================================
# KORAK 1: UČITAVANJE
# =============================================================

df = pd.read_csv('polise.csv', low_memory=False)


# =============================================================
# KORAK 2: FEATURE ENGINEERING (dijeli se između oba modela)
# =============================================================

def izvuci_starost(row):
    try:
        if pd.notna(row.get('datum_rodjenja')):
            god = pd.to_datetime(row['datum_rodjenja']).year
            return datetime.today().year - god
    except: pass
    try:
        jmbg = str(row.get('osig_jmbg_cli', ''))
        if len(jmbg) == 13 and jmbg.isdigit():
            god = int(jmbg[4:7])
            god = 1900 + god if god > 25 else 2000 + god
            return datetime.today().year - god
    except: pass
    return np.nan

def starost_grupa(s):
    if pd.isna(s):   return 'nepoznato'
    if s < 25:       return '18_24'
    if s < 35:       return '25_34'
    if s < 45:       return '35_44'
    if s < 55:       return '45_54'
    if s < 65:       return '55_64'
    return '65plus'

def premija_grupa(p):
    if pd.isna(p):    return 'nepoznato'
    if p < 100:       return 'niska'
    if p < 500:       return 'srednja'
    if p < 2000:      return 'visoka'
    return 'vrlo_visoka'

# Osnovni features
df['starost']       = df.apply(izvuci_starost, axis=1)
df['starost_grupa'] = df['starost'].apply(starost_grupa)
df['pol']           = df['pol_id'].map({1:'muski', 2:'zenski'}).fillna('nepoznato')
df['svojina']       = df['osig_svojina_cli'].map({1:'fizicko', 2:'pravno'}).fillna('nepoznato')
df['opstina']       = df['osig_opstina_cli'].fillna(0).astype(int).astype(str)
df['premija_grupa'] = df['premija_ukupno'].apply(premija_grupa)
df['ima_stetu']     = df['ind_steta'].fillna('N').apply(lambda x: 'ima' if x == 'D' else 'nema')
df['bracni']        = df['sif_bracni_status'].fillna('nepoznato').astype(str)
df['delatnost']     = df['sif_delatnost'].fillna('nepoznato').astype(str)
df['vrsta_polise']  = df['sif_vrsta'].astype(str)

# Vremenski features
df['dat_od'] = pd.to_datetime(df['dat_od'], errors='coerce')
df['godina_polise'] = df['dat_od'].dt.year.fillna(0).astype(int)

# Agregati po klijentu
agg = df.groupby('klijent_id').agg(
    ukupno_polisa    = ('vrsta_polise', 'count'),
    ukupna_premija   = ('premija_ukupno', 'sum'),
    prosjecna_premija= ('premija_ukupno', 'mean'),
    max_premija      = ('premija_ukupno', 'max'),
    broj_vrsta       = ('vrsta_polise', 'nunique'),
    ima_ikad_stetu   = ('ind_steta', lambda x: int('D' in x.values)),
    prva_polisa_god  = ('dat_od', lambda x: x.min().year if pd.notna(x.min()) else 0),
    zadnja_polisa_god= ('dat_od', lambda x: x.max().year if pd.notna(x.max()) else 0),
).reset_index()

agg['duzina_odnosa_god'] = datetime.today().year - agg['prva_polisa_god']

def aktivnost(n):
    if n == 1:  return 'novi'
    if n <= 3:  return 'aktivan'
    return 'loyalan'

agg['aktivnost'] = agg['ukupno_polisa'].apply(aktivnost)
df = df.merge(agg, on='klijent_id', how='left')

# Profil klijenta – jedan red po klijentu (zadnji poznati)
klijent_profil = (df.sort_values('dat_od')
                    .groupby('klijent_id')
                    .last()
                    .reset_index())

print(f"Klijenata:    {df['klijent_id'].nunique()}")
print(f"Vrsta polisa: {df['vrsta_polise'].nunique()}")
print(f"Interakcija:  {len(df)}")


# =============================================================
# KORAK 3: LIGHTFM MODEL
# =============================================================

print("\n--- LightFM ---")

dataset = Dataset()

user_feature_names = (
    ['pol_'        + x for x in df['pol'].unique()] +
    ['svojina_'    + x for x in df['svojina'].unique()] +
    ['starost_'    + x for x in df['starost_grupa'].unique()] +
    ['opstina_'    + x for x in df['opstina'].unique()] +
    ['premija_'    + x for x in df['premija_grupa'].unique()] +
    ['steta_'      + x for x in df['ima_stetu'].unique()] +
    ['bracni_'     + x for x in df['bracni'].unique()] +
    ['delatnost_'  + x for x in df['delatnost'].unique()] +
    ['aktivnost_'  + x for x in agg['aktivnost'].unique()]
)

dataset.fit(
    users=df['klijent_id'].unique(),
    items=df['vrsta_polise'].unique(),
    user_features=user_feature_names
)

# Interakcije
interactions, weights = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df.iterrows()
])

# User features
def napravi_uf(row):
    return [
        ('pol_'       + str(row['pol']),          1.0),
        ('svojina_'   + str(row['svojina']),       1.0),
        ('starost_'   + str(row['starost_grupa']), 1.0),
        ('opstina_'   + str(row['opstina']),       1.0),
        ('premija_'   + str(row['premija_grupa']), 1.0),
        ('steta_'     + str(row['ima_stetu']),     1.0),
        ('bracni_'    + str(row['bracni']),        1.0),
        ('delatnost_' + str(row['delatnost']),     1.0),
        ('aktivnost_' + str(row['aktivnost']),     1.0),
    ]

user_features = dataset.build_user_features([
    (row['klijent_id'], napravi_uf(row))
    for _, row in klijent_profil.iterrows()
])

# Train/test split po vremenu
split_datum = df['dat_od'].quantile(0.8)

train_inter, _ = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df[df['dat_od'] <= split_datum].iterrows()
])
test_inter, _ = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df[df['dat_od'] > split_datum].iterrows()
])

# Trening
lfm_model = LightFM(
    loss='warp',
    no_components=64,
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)

for epoch in range(30):
    lfm_model.fit_partial(
        train_inter,
        user_features=user_features,
        epochs=1, num_threads=4, verbose=False
    )
    if (epoch + 1) % 10 == 0:
        p = precision_at_k(lfm_model, train_inter,
                           user_features=user_features, k=3).mean()
        print(f"  Epoch {epoch+1}/30 | Precision@3 (train): {p:.4f}")

test_p   = precision_at_k(lfm_model, test_inter, train_inter,
                           user_features=user_features, k=3).mean()
test_auc = auc_score(lfm_model, test_inter, train_inter,
                     user_features=user_features).mean()
print(f"  Test Precision@3: {test_p:.4f} | AUC: {test_auc:.4f}")

# Mapiranja za predict
user_id_map, _, item_id_map, _ = dataset.mapping()
item_id_rev = {v: k for k, v in item_id_map.items()}
n_items = len(item_id_map)

def lfm_scores(klijent_id):
    """Vraća dict {vrsta_polise: lfm_score} za sve polise."""
    if klijent_id not in user_id_map:
        return {}
    uid = user_id_map[klijent_id]
    scores = lfm_model.predict(
        user_ids=uid,
        item_ids=np.arange(n_items),
        user_features=user_features
    )
    return {item_id_rev[i]: float(scores[i]) for i in range(n_items)}


# =============================================================
# KORAK 4: LGBM MODEL
# =============================================================
# Strategija: za svaku vrstu polise treniramo binarni klasifikator
# "Hoće li ovaj klijent uzeti ovu polisu?"
# Features: demografija + koje polise već ima + agregati
# =============================================================

print("\n--- LightGBM ---")

sve_vrste = df['vrsta_polise'].unique().tolist()

# Label encoderi za kategoričke
le_pol       = LabelEncoder().fit(df['pol'].fillna('nepoznato'))
le_svojina   = LabelEncoder().fit(df['svojina'].fillna('nepoznato'))
le_starost   = LabelEncoder().fit(df['starost_grupa'].fillna('nepoznato'))
le_bracni    = LabelEncoder().fit(df['bracni'].fillna('nepoznato'))
le_delatnost = LabelEncoder().fit(df['delatnost'].fillna('nepoznato'))
le_aktivnost = LabelEncoder().fit(agg['aktivnost'].fillna('nepoznato'))

# Sliding window dataset:
# Za svakog klijenta i svaku polisu u hronološkom nizu:
#   input  = sve polise PRIJE + demografija
#   target = da li je uzeo vrstu X nakon toga

primjeri = []

for kid, grupa in df.sort_values('dat_od').groupby('klijent_id'):
    vrste  = grupa['vrsta_polise'].tolist()
    profil = klijent_profil[klijent_profil['klijent_id'] == kid]
    if profil.empty:
        continue
    p = profil.iloc[0]

    for i in range(1, len(vrste)):
        prethodne = vrste[:i]
        ima_dict  = {f'ima_{v}': int(v in prethodne) for v in sve_vrste}

        primjer = {
            'klijent_id':        kid,
            'target_polisa':     vrste[i],
            'broj_prethodnih':   i,
            'starost':           p['starost'] if pd.notna(p['starost']) else -1,
            'starost_grupa_enc': le_starost.transform([str(p['starost_grupa'])])[0],
            'pol_enc':           le_pol.transform([str(p['pol'])])[0],
            'svojina_enc':       le_svojina.transform([str(p['svojina'])])[0],
            'bracni_enc':        le_bracni.transform([str(p['bracni'])])[0],
            'delatnost_enc':     le_delatnost.transform([str(p['delatnost'])])[0],
            'aktivnost_enc':     le_aktivnost.transform([str(p['aktivnost'])])[0],
            'ukupna_premija':    p['ukupna_premija'],
            'prosjecna_premija': p['prosjecna_premija'],
            'max_premija':       p['max_premija'],
            'broj_vrsta':        p['broj_vrsta'],
            'ima_ikad_stetu':    p['ima_ikad_stetu'],
            'duzina_odnosa_god': p['duzina_odnosa_god'],
            **ima_dict
        }
        primjeri.append(primjer)

df_primjeri = pd.DataFrame(primjeri).fillna(0)

feature_cols = (
    [f'ima_{v}' for v in sve_vrste] +
    [
        'broj_prethodnih', 'starost', 'starost_grupa_enc',
        'pol_enc', 'svojina_enc', 'bracni_enc', 'delatnost_enc',
        'aktivnost_enc', 'ukupna_premija', 'prosjecna_premija',
        'max_premija', 'broj_vrsta', 'ima_ikad_stetu', 'duzina_odnosa_god'
    ]
)

lgbm_modeli = {}

for vrsta in sve_vrste:
    y = (df_primjeri['target_polisa'] == vrsta).astype(int)
    if y.sum() < 20:
        continue

    X = df_primjeri[feature_cols]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"  Vrsta {vrsta}: AUC = {auc:.3f}")
    lgbm_modeli[vrsta] = model

def lgbm_scores(klijent_id):
    """Vraća dict {vrsta_polise: lgbm_proba} za sve polise."""
    k_polise = df[df['klijent_id'] == klijent_id]
    if k_polise.empty:
        return {}

    ima_polise = k_polise['vrsta_polise'].tolist()
    profil = klijent_profil[klijent_profil['klijent_id'] == klijent_id]
    if profil.empty:
        return {}
    p = profil.iloc[0]

    ima_dict = {f'ima_{v}': int(v in ima_polise) for v in sve_vrste}

    red = {
        'broj_prethodnih':   len(ima_polise),
        'starost':           p['starost'] if pd.notna(p['starost']) else -1,
        'starost_grupa_enc': le_starost.transform([str(p['starost_grupa'])])[0],
        'pol_enc':           le_pol.transform([str(p['pol'])])[0],
        'svojina_enc':       le_svojina.transform([str(p['svojina'])])[0],
        'bracni_enc':        le_bracni.transform([str(p['bracni'])])[0],
        'delatnost_enc':     le_delatnost.transform([str(p['delatnost'])])[0],
        'aktivnost_enc':     le_aktivnost.transform([str(p['aktivnost'])])[0],
        'ukupna_premija':    p['ukupna_premija'],
        'prosjecna_premija': p['prosjecna_premija'],
        'max_premija':       p['max_premija'],
        'broj_vrsta':        p['broj_vrsta'],
        'ima_ikad_stetu':    p['ima_ikad_stetu'],
        'duzina_odnosa_god': p['duzina_odnosa_god'],
        **ima_dict
    }

    X_pred = pd.DataFrame([red])[feature_cols]
    rezultat = {}
    for vrsta, model in lgbm_modeli.items():
        rezultat[vrsta] = float(model.predict_proba(X_pred)[0][1])
    return rezultat


# =============================================================
# KORAK 5: ENSEMBLE – kombinacija LightFM + LGBM
# =============================================================
# Težine: 0.4 LightFM + 0.6 LGBM
#
# Zašto 0.6 LGBM:
#   LGBM hvata nelinearne kombinacije featuresova
#   i direktno koristi demografiju → jači signal
#
# Zašto 0.4 LightFM:
#   Kolaborativni signal ("slični klijenti") dodaje
#   informaciju koju LGBM ne može sam da nauči
#
# Težine možeš tunirati na validacionom setu
# =============================================================

W_LFM  = 0.4
W_LGBM = 0.6

def min_max_normalize(scores_dict):
    """Normalizuje scoreove na [0, 1] da bi se mogli sabrati."""
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx == mn:
        return {k: 0.5 for k in scores_dict}
    return {k: (v - mn) / (mx - mn) for k, v in scores_dict.items()}

def ensemble_preporuke(klijent_id, top_n=3):
    """
    Kombinuje LightFM i LGBM score za finalnu preporuku.
    Vraća top N polisa koje klijent nema, rangirane po ensemble scoreu.
    """
    # Polise koje klijent već ima
    ima_polise = set(df[df['klijent_id'] == klijent_id]['vrsta_polise'].unique())

    # Scores iz oba modela – normalizovani na [0,1]
    lfm_s  = min_max_normalize(lfm_scores(klijent_id))
    lgbm_s = min_max_normalize(lgbm_scores(klijent_id))

    # Sve vrste polisa koje postoje
    sve = set(lfm_s.keys()) | set(lgbm_s.keys())

    ensemble = {}
    for vrsta in sve:
        if vrsta in ima_polise:
            continue  # preskači što već ima
        s_lfm  = lfm_s.get(vrsta, 0.0)
        s_lgbm = lgbm_s.get(vrsta, 0.0)
        ensemble[vrsta] = W_LFM * s_lfm + W_LGBM * s_lgbm

    rangirano = sorted(ensemble.items(), key=lambda x: x[1], reverse=True)
    return rangirano[:top_n]


# =============================================================
# KORAK 6: PRIMJER ZA JEDNOG KLIJENTA
# =============================================================

primjer_id = df['klijent_id'].iloc[0]
ima        = df[df['klijent_id'] == primjer_id]['vrsta_polise'].unique()
preporuke  = ensemble_preporuke(primjer_id)

print(f"\nKlijent: {primjer_id}")
print(f"Ima polise: {list(ima)}")
print(f"Preporuke (Ensemble LightFM + LGBM):")
for vrsta, score in preporuke:
    print(f"  → Vrsta {vrsta}  (ensemble score: {score:.3f})")


# =============================================================
# KORAK 7: APRIORI – objašnjenje za agenta
# =============================================================
# Model preporučuje, Apriori objašnjava ZAŠTO

from mlxtend.frequent_patterns import apriori, association_rules

pivot = df.pivot_table(
    index='klijent_id',
    columns='vrsta_polise',
    values='sif_vrsta',
    aggfunc='count'
).fillna(0).gt(0)

frequent_itemsets = apriori(pivot, min_support=0.03, use_colnames=True)
pravila = pd.DataFrame()

if not frequent_itemsets.empty:
    pravila = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
    pravila = pravila[pravila['confidence'] >= 0.3].sort_values('lift', ascending=False)
    print(f"\nTop Apriori pravila:")
    print(pravila[['antecedents','consequents','confidence','lift']].head(10).to_string(index=False))

def objasni(vrsta, pravila, ima_polise):
    if pravila.empty:
        return "Preporučeno na osnovu profila sličnih klijenata."
    rel = pravila[pravila['consequents'].apply(lambda x: vrsta in x)]
    for _, r in rel.iterrows():
        ant = set(r['antecedents'])
        if ant.issubset(set(ima_polise)):
            return (f"Od klijenata koji imaju {list(ant)}, "
                    f"{r['confidence']*100:.0f}% ima i polisu vrste {vrsta}.")
    return "Preporučeno na osnovu profila sličnih klijenata."

print(f"\nObjašnjenja za klijenta {primjer_id}:")
for vrsta, score in preporuke:
    print(f"  → Vrsta {vrsta}: {objasni(vrsta, pravila, list(ima))}")


# =============================================================
# KORAK 8: BATCH – preporuke za sve klijente
# =============================================================

print("\nGenerisanje preporuka za sve klijente...")

batch = []
for kid in df['klijent_id'].unique():
    try:
        preporuke = ensemble_preporuke(kid, top_n=3)
        ima_p     = list(df[df['klijent_id'] == kid]['vrsta_polise'].unique())
        for rang, (vrsta, score) in enumerate(preporuke, 1):
            batch.append({
                'klijent_id':            kid,
                'rang':                  rang,
                'preporuka_vrsta':       vrsta,
                'ensemble_score':        round(score, 4),
                'objasnjenje':           objasni(vrsta, pravila, ima_p)
            })
    except:
        continue

rezultati = pd.DataFrame(batch)
rezultati.to_csv('preporuke_ensemble.csv', index=False)

print(f"\n✓ Gotovo.")
print(f"  Preporuka: {len(rezultati)}")
print(f"  Klijenata: {rezultati['klijent_id'].nunique()}")
print(f"\nPrimjer izlaza:")
print(rezultati.head(9).to_string(index=False))
print("\nSačuvano: preporuke_ensemble.csv")


print(polise.groupby('klijent_id')['polisa_id'].count().describe())
```

Na osnovu tog rezultata mogu ti reći tačno koji model je optimalan. Jer:
```
Median = 1-2  →  LightFM + LGBM Ensemble  (trenutni kod)
Median = 3-4  →  Sliding window + LightFM + LGBM
Median = 5+   →  SASRec / Transformer dolazi u igru